# EV demand input data — `base/data`

Plots of all EV-related input files used by the Balmorel EV module in the **base** scenario:

| File | Content | Dimensions |
|---|---|---|
| `EV_BEV_available.inc` | BEV fleet size (number of vehicles) | year × region |
| `EV_PHEV_available.inc` | PHEV fleet size (currently all zero) | year × region |
| `EV_BEV_dumb.inc` (+ `EV_PHEV_dumb.inc`) | Share of vehicles using inflexible ("dumb") charging | year × region |
| `EV_BEV_return_profile{1-4}.inc` | Share of the fleet returning to a charger (connection profile) | year × week × hour-of-week × region |
| `EV_BEV_leave_profile{1-4}.inc` | Share of the fleet leaving the charger (departure profile) | year × week × hour-of-week × region |
| `EV_TECH_DATA.inc` | Charger capacity, efficiencies, battery size, degradation, cost, V2G share | year × parameter |

Profile variants (from the file headers): **1** = Monte Carlo per country; **2** = average of 1 (same for all countries);
**3** = like 1 but the EV returns to a charger after every trip; **4** = average of 3 (same for all countries).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm

# ---- settings --------------------------------------------------------------
DATA_DIR = Path('base/data')          # relative to the notebook (C:/Balmorel)
REGION   = 'DE4-E'                    # region used for the detailed profile plots
YEAR     = '2030'                     # profile year used for the detailed plots (2030/2040/2050)
APPLY_OVERRIDES = True                # EV_BEV_available.inc sets 2025 to 0 after the table; mimic that

# Okabe-Ito colour-blind-safe categorical palette, assigned in fixed order
PAL = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#D55E00', '#56B4E9', '#F0E442', '#000000']
mpl.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'axes.prop_cycle': mpl.cycler(color=PAL),
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})

## 1. Parsing helpers

GAMS `TABLE` blocks are parsed generically: comment lines (`*`) are skipped, the first remaining line is the column header,
and each following row is split into *labels* (year, `.`-separated set elements) and *values* (the last *n* tokens, *n* = number of columns).

In [ ]:
def read_gams_table(path, table=None):
    """Parse one GAMS TABLE block. Returns (label_rows, columns, values)."""
    lines = Path(path).read_text().splitlines()
    start = next(i for i, l in enumerate(lines)
                 if l.lstrip().upper().startswith('TABLE') and (table is None or table in l))
    body, cols = [], None
    for l in lines[start + 1:]:
        s = l.strip()
        if not s or s.startswith('*'):
            continue
        if s.startswith(';'):
            break
        if cols is None:
            cols = s.split()
            continue
        body.append(s.split())
    n = len(cols)
    labels, vals = [], np.empty((len(body), n))
    for i, r in enumerate(body):
        assert len(r) >= n + 1, f'{path}: row {i} has too few tokens'
        vals[i] = [float(x) for x in r[-n:]]
        labels.append([t for t in r[:-n] if t != '.'])
    return labels, cols, vals


def read_year_table(path, table=None):
    """Table with rows = year, columns = regions (or parameters)."""
    labels, cols, vals = read_gams_table(path, table)
    return pd.DataFrame(vals, index=[int(l[0]) for l in labels], columns=cols)


def read_profile(path):
    """Profile table: rows = year . week . hour-of-week, columns = regions."""
    labels, cols, vals = read_gams_table(path)
    idx = pd.MultiIndex.from_tuples(
        [(int(l[0]), l[1], int(l[2][1:])) for l in labels], names=['year', 'week', 'hour'])
    return pd.DataFrame(vals, index=idx, columns=cols)

## 2. Fleet size — `EV_BEV_available.inc`, `EV_PHEV_available.inc`

In [ ]:
bev = read_year_table(DATA_DIR / 'EV_BEV_available.inc', 'EV_BEV_available')
if APPLY_OVERRIDES and 2025 in bev.index:
    bev.loc[2025] = 0          # EV_BEV_available('2025',RRR)= 0;  at the bottom of the .inc file
phev = read_year_table(DATA_DIR / 'EV_PHEV_available.inc', 'EV_PHEV_available')

print(f'BEV : {bev.shape[1]} regions, years {list(bev.index)}')
print(f'PHEV: {phev.shape[1]} regions, years {list(phev.index)}, total = {phev.values.sum():,.0f} vehicles '
      f'(all zero: {bool((phev.values == 0).all())})')
bev.div(1e6).round(2).T.sort_values(bev.index[-1], ascending=False).head(12)   # million vehicles

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5.2), gridspec_kw={'width_ratios': [1, 1.6, 1.2]})

# (a) total fleet
tot = bev.sum(axis=1) / 1e6
ax[0].bar(tot.index.astype(str), tot.values, color=PAL[0], width=0.6)
for x, v in zip(tot.index.astype(str), tot.values):
    ax[0].text(x, v, f'{v:.0f}', ha='center', va='bottom', fontsize=8)
ax[0].set(title='(a) Total BEV fleet, all regions', ylabel='million vehicles')
ax[0].grid(axis='x', visible=False)

# (b) largest regions over time, direct-labelled; everything else in grey.
#     Regions with identical trajectories (e.g. the four DE4 regions) share one line and one label.
ranked = bev.iloc[-1].sort_values(ascending=False)
groups = {}
for r in ranked.index:
    groups.setdefault(tuple(bev[r].round(0)), []).append(r)
top_groups = list(groups.values())[:8]
shown = {r for g in top_groups for r in g}
for r in bev.columns.difference(list(shown)):
    ax[1].plot(bev.index, bev[r] / 1e6, color='0.8', lw=0.8, zorder=1)
for i, g in enumerate(top_groups):
    lab = g[0] if len(g) == 1 else '/'.join([g[0].split('-')[0]] + [x.split('-')[-1] for x in g]) if all('-' in x for x in g) else ', '.join(g)
    ax[1].plot(bev.index, bev[g[0]] / 1e6, color=PAL[i % len(PAL)], lw=2, marker='o', ms=4, label=lab, zorder=2)
    ax[1].annotate(lab, (bev.index[-1], bev[g[0]].iloc[-1] / 1e6), xytext=(4, 0), textcoords='offset points',
                   va='center', fontsize=8)
ax[1].set(title='(b) Largest regions (others grey)', ylabel='million vehicles',
          xlim=(bev.index[0], bev.index[-1] + 6))
ax[1].legend(ncol=2, fontsize=8, loc='upper left')

# (c) final-year ranking
last = bev.iloc[-1].sort_values()
last = last[last > 0]
ax[2].barh(last.index, last.values / 1e6, color=PAL[0], height=0.65)
ax[2].set(title=f'(c) BEV fleet in {bev.index[-1]} by region', xlabel='million vehicles')
ax[2].tick_params(axis='y', labelsize=7)
ax[2].grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

In [ ]:
# Region x year heat-map (log colour scale because fleets span several orders of magnitude)
h = bev.T.loc[bev.iloc[-1].sort_values(ascending=False).index]
h = h[(h > 0).any(axis=1)]
fig, ax = plt.subplots(figsize=(7, 0.22 * len(h) + 1.5))
im = ax.imshow(h.replace(0, np.nan).values, aspect='auto', cmap='Blues', norm=LogNorm())
ax.set_xticks(range(h.shape[1]), h.columns); ax.set_yticks(range(h.shape[0]), h.index, fontsize=7)
ax.grid(False)
for (i, j), v in np.ndenumerate(h.values):
    if v > 0:
        ax.text(j, i, f'{v/1e6:.2f}' if v >= 1e5 else f'{v/1e3:.0f}k', ha='center', va='center', fontsize=6,
                color='white' if v > h.values.max() / 30 else 'black')
ax.set_title('BEV fleet [million vehicles; thousands where < 0.1 M]')
plt.colorbar(im, ax=ax, label='vehicles (log)'); plt.tight_layout(); plt.show()

In [ ]:
# PHEV: all zero in the current base data ("BEV = all EVs"). Plotted anyway so a change in the file is noticed.
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(phev.index, phev.sum(axis=1) / 1e6, marker='o', color=PAL[1], lw=2)
ax.set(title='Total PHEV fleet (EV_PHEV_available.inc)', ylabel='million vehicles',
       ylim=(-0.05, max(1, phev.sum(axis=1).max() / 1e6)))
if (phev.values == 0).all():
    ax.text(0.5, 0.5, 'all zero', transform=ax.transAxes, ha='center', fontsize=16, color='0.5')
plt.tight_layout(); plt.show()

## 3. Dumb-charging share — `EV_BEV_dumb.inc`, `EV_PHEV_dumb.inc`

Share of the fleet that charges immediately on arrival (inflexible). The remainder is smart/flexible charging.

In [ ]:
dumb = {'BEV': read_year_table(DATA_DIR / 'EV_BEV_dumb.inc', 'EV_BEV_dumb'),
        'PHEV': read_year_table(DATA_DIR / 'EV_PHEV_dumb.inc', 'EV_PHEV_dumb')}
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, (k, d) in zip(axs, dumb.items()):
    same = (d.nunique(axis=1) == 1).all()          # identical across regions in every year?
    if same:
        a.plot(d.index, d.iloc[:, 0], marker='o', color=PAL[0], lw=2, label=f'all {d.shape[1]} regions identical')
    else:
        for r in d.columns:
            a.plot(d.index, d[r], lw=1.2, label=r)
    a.set(title=f'{k} dumb-charging share', xlabel='year')
    a.legend(fontsize=8, ncol=1 if same else 2)
axs[0].set_ylabel('share of fleet')
plt.tight_layout(); plt.show()
dumb['BEV'].iloc[:, :6]

## 4. Connection profiles — `EV_BEV_return_profile*.inc`, `EV_BEV_leave_profile*.inc`

Each profile has 3 years (2030, 2040, 2050) × 52 weeks (`S01`–`S52`) × 168 hours of the week (`T001`–`T168`) × region.
`return` = share of the fleet plugging in at that hour; `leave` = share unplugging (stored with a negative sign in the files; plotted here as a positive magnitude).

In [ ]:
PROFILES = {}   # (kind, fleet, variant) -> DataFrame
for kind in ['return', 'leave']:
    for v in [1, 2, 3, 4]:
        PROFILES[(kind, 'BEV', v)] = read_profile(DATA_DIR / f'EV_BEV_{kind}_profile{v}.inc')
    PROFILES[(kind, 'PHEV', 1)] = read_profile(DATA_DIR / f'EV_PHEV_{kind}_profile1.inc')

# The leave profiles are stored with a NEGATIVE sign in the .inc files (outflow from the connected fleet).
# Record that, then flip to magnitudes so return and leave are directly comparable in the plots below.
LEAVE_STORED_NEGATIVE = all((d.values <= 0).all() for k, d in PROFILES.items() if k[0] == 'leave')
print('leave profiles stored as <= 0 in the files:', LEAVE_STORED_NEGATIVE)
for k in PROFILES:
    if k[0] == 'leave':
        PROFILES[k] = -PROFILES[k]

summary = pd.DataFrame({
    f'{k[1]} {k[0]} p{k[2]}': {'regions': d.shape[1], 'weeks': d.index.get_level_values('week').nunique(),
                               'max value': d.values.max(),
                               'mean weekly sum': d.groupby(['year', 'week']).sum().values.mean()}
    for k, d in PROFILES.items()}).T
summary

In [ ]:
DAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']   # assumes hour 1 = Monday 00:00; adjust if the model uses another convention

def week_axis(a):
    a.set_xlim(1, 168)
    a.set_xticks(np.arange(13, 169, 24), DAYS)
    for x in range(25, 169, 24):
        a.axvline(x, color='0.7', lw=0.5)

def mean_profile(df, year):
    "Average over the 52 weeks -> 168 x regions."
    return df.loc[year].groupby('hour').mean()

# return and leave for every variant: cross-region mean with min-max band
fig, axs = plt.subplots(2, 4, figsize=(18, 7), sharex=True)
for row, kind in enumerate(['return', 'leave']):
    for col, v in enumerate([1, 2, 3, 4]):
        mp = mean_profile(PROFILES[(kind, 'BEV', v)], int(YEAR))
        a = axs[row, col]
        a.fill_between(mp.index, mp.min(axis=1), mp.max(axis=1), color=PAL[5], alpha=0.35, label='min-max over regions')
        a.plot(mp.index, mp.mean(axis=1), color=PAL[0], lw=1.4, label='mean over regions')
        a.plot(mp.index, mp[REGION], color=PAL[4], lw=1.0, label=REGION)
        a.set_title(f'{kind} - profile {v}'); week_axis(a)
axs[0, 0].legend(fontsize=7)
axs[0, 0].set_ylabel('share of fleet / hour'); axs[1, 0].set_ylabel('share of fleet / hour')
fig.suptitle(f'BEV connection profiles, year {YEAR}, mean over 52 weeks', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Compare variants directly for one region
fig, axs = plt.subplots(1, 2, figsize=(15, 4), sharex=True, sharey=True)
for a, kind in zip(axs, ['return', 'leave']):
    for i, v in enumerate([1, 2, 3, 4]):
        mp = mean_profile(PROFILES[(kind, 'BEV', v)], int(YEAR))[REGION]
        a.plot(mp.index, mp.values, color=PAL[i], lw=1.3, label=f'profile {v}')
    a.set_title(f'{kind} profile - {REGION}, {YEAR}'); week_axis(a); a.legend(fontsize=8, ncol=4)
axs[0].set_ylabel('share of fleet / hour')
plt.tight_layout(); plt.show()

# Cumulative balance (return - leave) over the week for variants 1 and 3
fig, a = plt.subplots(figsize=(10, 3.5))
for i, v in enumerate([1, 3]):
    r = mean_profile(PROFILES[('return', 'BEV', v)], int(YEAR))[REGION]
    l = mean_profile(PROFILES[('leave', 'BEV', v)], int(YEAR))[REGION]
    a.plot(r.index, (r - l).cumsum(), color=PAL[i], lw=1.5, label=f'profile {v}')
a.axhline(0, color='k', lw=0.6); week_axis(a); a.legend()
a.set(title=f'Cumulative (return - leave), {REGION}, {YEAR}', ylabel='cumulative share')
plt.tight_layout(); plt.show()

In [ ]:
# Week x hour heat-maps (all 52 weeks) - does the profile vary through the year?
fig, axs = plt.subplots(2, 2, figsize=(15, 6.5), sharex=True)
for row, kind in enumerate(['return', 'leave']):
    for col, v in enumerate([1, 3]):
        d = PROFILES[(kind, 'BEV', v)].loc[int(YEAR), REGION].unstack('hour')
        im = axs[row, col].imshow(d.values, aspect='auto', cmap='viridis', extent=[0.5, 168.5, d.shape[0] + 0.5, 0.5])
        axs[row, col].set_title(f'{kind} - profile {v} ({REGION}, {YEAR})'); axs[row, col].grid(False)
        axs[row, col].set_ylabel('week'); plt.colorbar(im, ax=axs[row, col], pad=0.01)
for a in axs[1]:
    a.set_xticks(np.arange(13, 169, 24), DAYS)
plt.tight_layout(); plt.show()

In [ ]:
# Differences between years and between regions (return profile 1)
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
years = sorted(PROFILES[('return', 'BEV', 1)].index.get_level_values('year').unique())
for i, y in enumerate(years):
    mp = mean_profile(PROFILES[('return', 'BEV', 1)], y)[REGION]
    axs[0].plot(mp.index, mp.values, color=PAL[i], lw=1.3, label=str(y))
axs[0].set_title(f'Return profile 1 by year - {REGION}'); week_axis(axs[0]); axs[0].legend()

mp = mean_profile(PROFILES[('return', 'BEV', 1)], int(YEAR))
wk = mp.sum().sort_values()
axs[1].bar(range(len(wk)), wk.values - wk.mean(), color=PAL[0])
axs[1].set_xticks(range(len(wk)), wk.index, rotation=90, fontsize=7)
axs[1].set(title=f'Return profile 1 - weekly sum per region minus cross-region mean ({YEAR})', ylabel='deviation')
axs[1].grid(axis='x', visible=False)
plt.tight_layout(); plt.show()

In [ ]:
# PHEV profile 1 (only variant provided) vs BEV profile 1
fig, axs = plt.subplots(1, 2, figsize=(15, 3.8), sharex=True, sharey=True)
for a, kind in zip(axs, ['return', 'leave']):
    for i, fl in enumerate(['BEV', 'PHEV']):
        mp = mean_profile(PROFILES[(kind, fl, 1)], int(YEAR))[REGION]
        a.plot(mp.index, mp.values, color=PAL[i], lw=1.3, ls='-' if fl == 'BEV' else '--', label=fl)
    a.set_title(f'{kind} profile 1 - {REGION}, {YEAR}'); week_axis(a); a.legend()
plt.tight_layout(); plt.show()
print('PHEV profile identical to BEV profile 1:',
      all(np.allclose(PROFILES[(k, 'PHEV', 1)].values, PROFILES[(k, 'BEV', 1)].values) for k in ['return', 'leave']))

## 5. Technical data — `EV_TECH_DATA.inc`

Two tables (`BEV_TECH_DATA`, `PHEV_TECH_DATA`) with one row per year. Note from the file: *PHEV is not up to date; all EVs are accumulated in BEV.*

In [ ]:
tech = {'BEV': read_year_table(DATA_DIR / 'EV_TECH_DATA.inc', 'BEV_TECH_DATA'),
        'PHEV': read_year_table(DATA_DIR / 'EV_TECH_DATA.inc', 'PHEV_TECH_DATA')}
UNITS = {'EV_chargercap': 'MWh/h per vehicle', 'EV_return_energy': 'MWh on return', 'EV_leave_energy': 'MWh on departure',
         'EV_G2V_PEFF': 'charging efficiency', 'EV_V2G_PEFF': 'discharging efficiency',
         'EV_degredation_cyclic': 'degradation per cycle', 'EV_dumb_energy': 'MWh per dumb charge',
         'EV_batsize': 'MWh', 'EV_emergency': 'MWh reserve', 'EV_batcost': 'EUR/kWh', 'Share_EV_V2G': 'share of fleet'}
tech['BEV'].T

In [ ]:
params = list(tech['BEV'].columns)
ncol = 4; nrow = int(np.ceil(len(params) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(17, 3.1 * nrow))
for a, p in zip(axs.flat, params):
    a.plot(tech['BEV'].index, tech['BEV'][p], marker='o', color=PAL[0], lw=2, label='BEV')
    if p in tech['PHEV']:
        a.plot(tech['PHEV'].index, tech['PHEV'][p], marker='s', ms=4, color=PAL[1], lw=1.5, ls='--', label='PHEV')
    a.set_title(p, fontsize=9); a.set_ylabel(UNITS.get(p, ''), fontsize=8)
    lo, hi = a.get_ylim(); pad = (hi - lo) * 0.15 or abs(hi) * 0.1 or 0.1
    a.set_ylim(lo - pad, hi + pad)                       # keep flat series readable
    a.set_xticks(tech['BEV'].index[::2])
for a in list(axs.flat)[len(params):]:
    a.set_visible(False)
axs.flat[0].legend(fontsize=8)
fig.suptitle('EV technical assumptions (EV_TECH_DATA.inc)', fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Interpolated profiles for missing years

The profile files only contain 2030, 2040 and 2050, while the model also simulates e.g. 2035. `EV_pardefine.inc` therefore fills every year without data by
**linear interpolation in the year** between the nearest earlier and later year with data (no extrapolation outside the data range). This section re-implements that rule
and plots the result; interpolated years are drawn dashed.

In [ ]:
MODEL_YEARS = [2025, 2030, 2035, 2040, 2045, 2050]   # years to show / fill (2035 is the one simulated in base_dc)
VARIANT = 1                                          # profile variant to plot (1-4)

def interpolate_years(df, years):
    "Mirror of the GAMS rule in EV_pardefine.inc. Returns (df_with_extra_years, {year: (lo, hi, w)})."
    have = sorted(df.index.get_level_values('year').unique())
    parts, info = [df], {}
    for y in years:
        if y in have:
            continue
        lo = max((h for h in have if h < y), default=None)
        hi = min((h for h in have if h > y), default=None)
        if lo is None or hi is None:             # outside the data range: not extrapolated (stays 0 / absent)
            continue
        w = (y - lo) / (hi - lo)
        blk = df.loc[lo].copy()
        blk.iloc[:, :] = (1 - w) * df.loc[lo].values + w * df.loc[hi].values
        blk.index = pd.MultiIndex.from_tuples([(y, wk, h) for wk, h in blk.index], names=df.index.names)
        parts.append(blk); info[y] = (lo, hi, w)
    return pd.concat(parts).sort_index(), info

INTERP = {}
for kind in ['return', 'leave']:
    INTERP[kind], info = interpolate_years(PROFILES[(kind, 'BEV', VARIANT)], MODEL_YEARS)
print(f'Profile variant {VARIANT}; interpolated years (year: lower, upper, weight on upper):', info)
data_years = sorted(PROFILES[('return', 'BEV', VARIANT)].index.get_level_values('year').unique())
skipped = [y for y in MODEL_YEARS if y not in data_years and y not in info]
print('Years with no data and no interpolation possible (stay 0 in the model):', skipped)

# check: 2035 must be exactly the midpoint of 2030 and 2040
if 2035 in info:
    for kind in ['return', 'leave']:
        d = INTERP[kind]
        err = np.abs(d.loc[2035].values - 0.5 * (d.loc[2030].values + d.loc[2040].values)).max()
        print(f'{kind}: max |2035 - mean(2030, 2040)| = {err:.2e}')

In [ ]:
# Weekly pattern for one region: data years solid, interpolated years dashed
yrs_shown = [y for y in MODEL_YEARS if y in INTERP['return'].index.get_level_values('year')]
cmap = {y: PAL[i % len(PAL)] for i, y in enumerate(yrs_shown)}
fig, axs = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
for a, kind in zip(axs, ['return', 'leave']):
    for y in yrs_shown:
        mp = mean_profile(INTERP[kind], y)[REGION]
        a.plot(mp.index, mp.values, color=cmap[y], lw=1.4, ls='-' if y in data_years else '--',
               label=f'{y}' + ('' if y in data_years else ' (interpolated)'))
    a.set_title(f'{kind} profile {VARIANT} - {REGION}, mean over weeks'); week_axis(a)
    a.set_ylabel('share of fleet / hour')
axs[0].legend(ncol=5, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# The profiles are nearly identical across years, so show the DIFFERENCE to the first data year to make the interpolation visible
base_y = data_years[0]
fig, axs = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
for a, kind in zip(axs, ['return', 'leave']):
    ref = mean_profile(INTERP[kind], base_y)[REGION]
    for y in yrs_shown:
        if y == base_y:
            continue
        mp = mean_profile(INTERP[kind], y)[REGION]
        a.plot(mp.index, mp.values - ref.values, color=cmap[y], lw=1.2, ls='-' if y in data_years else '--',
               label=f'{y} - {base_y}' + ('' if y in data_years else ' (interpolated)'))
    a.axhline(0, color='k', lw=0.6); week_axis(a)
    a.set_title(f'{kind} profile {VARIANT} - {REGION}: difference to {base_y}'); a.set_ylabel('difference in share')
axs[0].legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Weekly total (return / leave) across years: mean over regions with min-max band. Interpolated years = open markers.
fig, axs = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for a, kind in zip(axs, ['return', 'leave']):
    d = INTERP[kind]
    wk = d.groupby(['year', 'week']).sum().groupby('year').mean()          # weekly sum per region, mean over weeks
    ys = wk.index.values
    a.fill_between(ys, wk.min(axis=1), wk.max(axis=1), color=PAL[5], alpha=0.3, label='min-max over regions')
    a.plot(ys, wk.mean(axis=1), color=PAL[0], lw=1.5, label='mean over regions')
    a.plot(ys, wk[REGION], color=PAL[4], lw=1.2, label=REGION)
    is_int = [y not in data_years for y in ys]
    a.scatter(ys, wk[REGION], s=40, facecolors=['white' if i else PAL[4] for i in is_int], edgecolors=PAL[4], zorder=3)
    a.set(title=f'{kind} profile {VARIANT}: weekly sum of fleet share', xlabel='year', ylabel='sum over 168 h')
    a.set_xticks(ys); a.legend(fontsize=8)
fig.suptitle('Open markers = interpolated years', y=1.0, fontsize=9)
plt.tight_layout(); plt.show()

## 7. Quick data-quality checks

In [ ]:
checks = {
    'leave profiles stored as <= 0 in the files': LEAVE_STORED_NEGATIVE,
    'BEV available >= 0': bool((bev.values >= 0).all()),
    'PHEV available all zero': bool((phev.values == 0).all()),
    'dumb share within [0, 1]': all(bool(((d.values >= 0) & (d.values <= 1)).all()) for d in dumb.values()),
    'profile magnitudes within [0, 1] (leave sign-flipped)': all(bool(((d.values >= 0) & (d.values <= 1)).all()) for d in PROFILES.values()),
    'BEV regions with fleet but no profile': sorted(set(bev.columns[(bev > 0).any()]) - set(PROFILES[('return', 'BEV', 1)].columns)),
    'dumb-share regions missing from BEV fleet table': sorted(set(dumb['BEV'].columns) - set(bev.columns)),
}
for k, v in checks.items():
    print(f'{k:50s} {v}')